In [23]:
#imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import joblib 

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [24]:
#Load Data
df = pd.read_csv('pipe_condition_class_synthetic.csv')
print(df.head())
df.info()

#Create Features and Target Variable
x = df.drop('Condition Rating', axis = 1)
y = df['Condition Rating']


   Age Material  Diameter  Slope  Depth      Length  Soil PH Soil Type  \
0   57      VCP         8   0.34  11.32  221.045734      5.7      Rock   
1   35      VCP        12   0.40   7.00  342.975986      6.8      Clay   
2   28      PVC         8   0.50   8.58  295.072420      5.4      Sand   
3   31      PVC        15   0.17  10.50  492.085601      8.2      Clay   
4   58      PVC         8   0.36   8.58  334.559947      8.2      Clay   

  Road Type  Condition Rating  
0    Street                 3  
1    Street                 5  
2    Street                 1  
3    Street                 1  
4     Alley                 3  
<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Age               30000 non-null  int64  
 1   Material          30000 non-null  str    
 2   Diameter          30000 non-null  int64  
 3   Slope             30000 n

In [25]:
#Split Categorical and Numerical Features
numerical_cols = ['Age','Diameter','Slope','Depth','Length','Soil PH']
categorical_cols = ['Material','Road Type','Soil Type']

#Training and Testing Datasets
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)

Training features shape: (24000, 9)
Testing features shape: (6000, 9)


In [26]:
#Imputation & Encoding
numerical_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy= 'median')),
    ('scaler',StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')), 
    ('encoder',OneHotEncoder(handle_unknown='ignore'))
])

#Join Pipeline
preprocessor = ColumnTransformer([
    ('num',numerical_pipeline,numerical_cols),
    ('cat',categorical_pipeline,categorical_cols)
])

pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',RandomForestClassifier(class_weight='balanced', random_state=42))
])

In [28]:
#Train and Model
pipeline.fit(X_train,y_train)
prediction = pipeline.predict(X_test)

#View Encoding
encoded_cols = pipeline.named_steps['preprocessor'].transformers_[1][1].named_steps['encoder'].get_feature_names_out(categorical_cols)
print("Encoded Categorical Columns:", encoded_cols)

#Evaluate Model
print("\nAccuracy:", accuracy_score(y_test, prediction))
print("Classification Report:")
print(classification_report(y_test, prediction))

Encoded Categorical Columns: ['Material_PVC' 'Material_RC' 'Material_VCP' 'Road Type_Alley'
 'Road Type_Easement' 'Road Type_Highway' 'Road Type_Street'
 'Soil Type_Clay' 'Soil Type_Gravel' 'Soil Type_Loam' 'Soil Type_Rock'
 'Soil Type_Sand']

Accuracy: 0.5723333333333334
Classification Report:
              precision    recall  f1-score   support

           1       0.62      0.75      0.68      2958
           2       0.00      0.00      0.00       265
           3       0.50      0.53      0.51      2289
           4       0.15      0.01      0.02       221
           5       0.24      0.01      0.03       267

    accuracy                           0.57      6000
   macro avg       0.30      0.26      0.25      6000
weighted avg       0.51      0.57      0.53      6000

